# RL for Language Models - Component Demo

This notebook demonstrates how each component of our RL system works with example inputs and outputs.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

/Users/ammarh/Documents/second-brain-2/ai-ml/code-junkyard/rl_llm_tutorial/myrl_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Value Head Component

The value head estimates the expected return from a given state.

In [3]:
class ValueHead(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        return self.value_head(hidden_states)

# Demo with random inputs
batch_size = 2
seq_len = 4
hidden_size = 8

# Create random hidden states
hidden_states = torch.randn(batch_size, seq_len, hidden_size)

# Initialize value head
value_head = ValueHead(hidden_size)

# Get value estimates
values = value_head(hidden_states)

print("Input hidden states shape:", hidden_states.shape)
print("Output values shape:", values.shape)
print("\nExample values:")
print(values)

Input hidden states shape: torch.Size([2, 4, 8])
Output values shape: torch.Size([2, 4, 1])

Example values:
tensor([[[-0.2438],
         [-0.3699],
         [-0.6056],
         [-0.4259]],

        [[-0.4962],
         [-0.6083],
         [-0.3535],
         [-0.4524]]], grad_fn=<ViewBackward0>)


## 2. Policy Network (Language Model)

The policy network is our language model that takes prompts and generates responses.

In [14]:
class RLModel(nn.Module):
    def __init__(self, model_name: str = "microsoft/phi-1_5"):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Add padding token if it doesn't exist
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        # Initialize value head
        self.value_head = ValueHead(self.model.config.hidden_size)
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        return_value: bool = True
    ) -> Dict[str, torch.Tensor]:
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=return_value,
            return_dict=True
        )
        # for key in outputs.keys():  # This is equivalent to: for key in my_dict.keys():
        #     print(f"key -- {type(outputs[key])}")
        # print(outputs['past_key_values'])
        # for i in range(len(outputs['hidden_states'])):
        #     print(outputs['hidden_states'][i].shape)

        result = {"logits": outputs.logits}
        
        if return_value:
            values = self.value_head(outputs.hidden_states[-1])
            result["values"] = values
            
        return result
    
    def generate_with_value(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        max_length: int = 512,
        temperature: float = 1.0,
        **kwargs
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Generate text
        generated_ids = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            do_sample=True,
            temperature=temperature,
            pad_token_id=self.tokenizer.pad_token_id,
            **kwargs
        )
        
        # Compute values for the generated sequence
        with torch.no_grad():
            outputs = self.model(
                input_ids=generated_ids,
                output_hidden_states=True,
                return_dict=True
            )
            values = self.value_head(outputs.hidden_states[-1])
            
        return generated_ids, values

# Initialize the model
model = RLModel(model_name="HuggingFaceTB/SmolLM2-135M-Instruct")

# Example prompts
prompts = [
    "Write a short story about a cat:",
    "Explain quantum computing:"
]

# Tokenize prompts
inputs = model.tokenizer(
    prompts,
    padding=True,
    return_tensors="pt"
)

print("Input shapes:")
print(f"Input IDs: {inputs.input_ids.shape} ... {inputs.input_ids}")
print(f"Attention mask: {inputs.attention_mask.shape} ... {inputs.attention_mask}")

# Get model outputs
with torch.no_grad():
    outputs = model(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask
    )

print("\nOutput shapes:")
print(f"Logits: {outputs['logits'].shape}")
print(f"Values: {outputs['values'].shape}")

# Generate responses
generated_ids, values = model.generate_with_value(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_length=100,
    temperature=0.7
)

print(generated_ids.shape)
print(values.shape)

# Decode generated text
generated_texts = model.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

print("\nGenerated responses:")
for prompt, response in zip(prompts, generated_texts):
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response}")

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Input shapes:
Input IDs: torch.Size([2, 8]) ... tensor([[19161,   253,  1890,  1977,   563,   253,  2644,    42],
        [36971,  8671,  7867,    42,     2,     2,     2,     2]])
Attention mask: torch.Size([2, 8]) ... tensor([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0]])

Output shapes:
Logits: torch.Size([2, 8, 49152])
Values: torch.Size([2, 8, 1])
torch.Size([2, 100])
torch.Size([2, 100, 1])

Generated responses:

Prompt: Write a short story about a cat:
Response: Write a short story about a cat:

<<The Cat Who Haunts the Park>>

The sun was shining in the bright sky, and the world was alive with the sounds of birds chirping and the rustling of leaves in the breeze. It was a day that seemed to bring all the magic and wonder to the world.

Every morning, the cat would run from one end of the park to the other, always looking up at the sky and watching the world go

Prompt: Explain quantum computing:
Response: Explain quantum computing:

Quantum computing operates at t

In [11]:
model.model.config

LlamaConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": 2,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_scaling": null,
  "rope_theta": 100000,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers.js_config": {
    "kv_cache_dtype": {
      "fp16": "float16",
      "q4f16": "float16"
    }
  },
  "transformers_version": "4.51.3",
  "use_cache": true,
  "vocab_size": 49152
}

In [16]:
# from torchinfo import summary
# summary(model.model, input_size=(1, 512))  # Adjust input size based on your model


# Print layer by layer
for name, module in model.model.named_modules():
    if name:  # Skip the model itself
        print(f"{name}: {module.__class__.__name__}")

model: LlamaModel
model.embed_tokens: Embedding
model.layers: ModuleList
model.layers.0: LlamaDecoderLayer
model.layers.0.self_attn: LlamaAttention
model.layers.0.self_attn.q_proj: Linear
model.layers.0.self_attn.k_proj: Linear
model.layers.0.self_attn.v_proj: Linear
model.layers.0.self_attn.o_proj: Linear
model.layers.0.mlp: LlamaMLP
model.layers.0.mlp.gate_proj: Linear
model.layers.0.mlp.up_proj: Linear
model.layers.0.mlp.down_proj: Linear
model.layers.0.mlp.act_fn: SiLU
model.layers.0.input_layernorm: LlamaRMSNorm
model.layers.0.post_attention_layernorm: LlamaRMSNorm
model.layers.1: LlamaDecoderLayer
model.layers.1.self_attn: LlamaAttention
model.layers.1.self_attn.q_proj: Linear
model.layers.1.self_attn.k_proj: Linear
model.layers.1.self_attn.v_proj: Linear
model.layers.1.self_attn.o_proj: Linear
model.layers.1.mlp: LlamaMLP
model.layers.1.mlp.gate_proj: Linear
model.layers.1.mlp.up_proj: Linear
model.layers.1.mlp.down_proj: Linear
model.layers.1.mlp.act_fn: SiLU
model.layers.1.inp

## 3. Experience Container and GAE

Here we demonstrate how experiences are stored and how advantages are computed.

In [15]:
@dataclass
class Experience:
    state_ids: torch.Tensor
    action_ids: torch.Tensor
    attention_mask: torch.Tensor
    rewards: torch.Tensor
    values: torch.Tensor
    log_probs: torch.Tensor
    advantages: torch.Tensor = None
    returns: torch.Tensor = None

def compute_gae(
    rewards: torch.Tensor,
    values: torch.Tensor,
    gamma: float = 0.99,
    lambda_: float = 0.95
) -> torch.Tensor:
    advantages = torch.zeros_like(rewards)
    last_gae = 0
    
    for t in reversed(range(len(rewards))):
        next_value = values[t + 1] if t < len(rewards) - 1 else 0
        delta = rewards[t] + gamma * next_value - values[t]
        advantages[t] = delta + gamma * lambda_ * last_gae
        last_gae = advantages[t]
        
    return advantages

# Example with synthetic data
rewards = torch.tensor([1.0, 0.0, 2.0, -1.0, 3.0])
values = torch.tensor([0.5, 0.8, 1.2, 0.3, 2.0, 0.0])  # Include final value

advantages = compute_gae(rewards, values)

print("Rewards:", rewards)
print("Values:", values[:-1])  # Exclude final value
print("Computed advantages:", advantages)

Rewards: tensor([ 1.,  0.,  2., -1.,  3.])
Values: tensor([0.5000, 0.8000, 1.2000, 0.3000, 2.0000])
Computed advantages: tensor([3.9754, 2.8531, 2.6211, 1.6205, 1.0000])


## 4. Processing Text Through the Policy

In [16]:
def process_text_with_policy(
    model: RLModel,
    text: str,
    max_length: int = 100,
    temperature: float = 1.0,
    return_values: bool = True
):
    # Tokenize input
    inputs = model.tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    
    # Get policy outputs (logits and values)
    with torch.no_grad():
        policy_outputs = model(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            return_value=return_values
        )
    
    # Get next token probabilities
    next_token_logits = policy_outputs["logits"][:, -1, :]
    next_token_probs = F.softmax(next_token_logits / temperature, dim=-1)
    
    # Sample next tokens
    next_tokens = torch.multinomial(next_token_probs, num_samples=1)
    
    # Generate full response
    generated_ids, values = model.generate_with_value(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_length,
        temperature=temperature
    )
    
    # Decode response
    response = model.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    return {
        "input_tokens": inputs.input_ids[0],
        "input_text": text,
        "response": response,
        "next_token_probs": next_token_probs[0],
        "values": values[0] if return_values else None,
        "policy_outputs": policy_outputs
    }

# Example usage
example_texts = [
    "Translate this to French: Hello, how are you?",
    "Write a haiku about spring:",
    "Explain the concept of recursion:"
]

for text in example_texts:
    print(f"\nProcessing: {text}")
    results = process_text_with_policy(model, text, temperature=0.7)
    
    print("\nResponse:")
    print(results["response"])
    
    print("\nTop 5 next token probabilities:")
    probs, indices = results["next_token_probs"].topk(5)
    for prob, idx in zip(probs, indices):
        token = model.tokenizer.decode([idx])
        print(f"{token}: {prob:.3f}")
    
    if results["values"] is not None:
        print("\nValue estimates:")
        print(results["values"][:5])  # Show first 5 values


Processing: Translate this to French: Hello, how are you?

Response:
Translate this to French: Hello, how are you?

In English, this phrase is: "Hello, how are you?"

This sentence structure is how the English sentence "Hello, how are you?" is interpreted in French.

Top 5 next token probabilities:

: 0.685
<|im_end|>: 0.193
 I: 0.026
 Hello: 0.022
 
: 0.011

Value estimates:
tensor([[ 0.2756],
        [-0.0357],
        [ 0.0447],
        [-0.1438],
        [-0.0317]])

Processing: Write a haiku about spring:

Response:
Write a haiku about spring:

"Roses bloom in the moon's pale light,
Spring's gentle scent fills the air,
In spring, the sun beams down,
Nature's return to life."

This haiku captures the essence of spring's transformation and renewal, inviting the reader to immerse themselves in nature's beauty.

Top 5 next token probabilities:

: 0.954


: 0.033
 ": 0.005

  : 0.005
 
: 0.001

Value estimates:
tensor([[ 0.2201],
        [-0.2314],
        [ 0.1055],
        [ 0.3225]

## 5. Generation Strategies

In [17]:
def generate_with_strategies(
    model: RLModel,
    text: str,
    strategies: List[Dict]
):
    results = {}
    
    # Tokenize input once
    inputs = model.tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )
    
    for strategy in strategies:
        name = strategy.pop("name")
        print(f"\nGenerating with {name}:")
        
        # Generate with current strategy
        generated_ids, values = model.generate_with_value(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            **strategy
        )
        
        # Decode response
        response = model.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        
        results[name] = {
            "response": response,
            "values": values[0]
        }
        
        print(response)
        print(f"Average value estimate: {values.mean().item():.3f}")
    
    return results

# Example strategies
strategies = [
    {
        "name": "Greedy",
        "do_sample": False,
        "max_length": 100
    },
    {
        "name": "Sampling (T=0.7)",
        "do_sample": True,
        "temperature": 0.7,
        "max_length": 100
    },
    {
        "name": "Nucleus Sampling (p=0.9)",
        "do_sample": True,
        "top_p": 0.9,
        "max_length": 100
    }
]

example_text = "Write a creative story about time travel:"
results = generate_with_strategies(model, example_text, strategies)


Generating with Greedy:


TypeError: transformers.generation.utils.GenerationMixin.generate() got multiple values for keyword argument 'do_sample'

## 6. Policy Analysis

In [19]:
def analyze_policy_behavior(
    model: RLModel,
    text: str,
    n_samples: int = 5,
    temperature: float = 0.7
):
    print(f"Analyzing policy behavior for: {text}\n")
    
    # Tokenize input
    inputs = model.tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )
    
    # Get base policy outputs
    with torch.no_grad():
        outputs = model(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask
        )
    
    # 1. Token Distribution Analysis
    next_token_logits = outputs["logits"][0, -1, :]
    next_token_probs = F.softmax(next_token_logits / temperature, dim=-1)
    
    print("Top 10 next token probabilities:")
    probs, indices = next_token_probs.topk(10)
    for prob, idx in zip(probs, indices):
        token = model.tokenizer.decode([idx])
        print(f"{token}: {prob:.3f}")
    
    # 2. Generate multiple samples
    print(f"\n{n_samples} different completions:")
    for i in range(n_samples):
        generated_ids, values = model.generate_with_value(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=100,
            temperature=temperature,
            # do_sample=True
        )
        
        response = model.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        avg_value = values.mean().item()
        
        print(f"\nSample {i+1} (avg value: {avg_value:.3f}):")
        print(response)
    
    # 3. Value distribution
    print("\nValue distribution for generated samples:")
    print(f"Mean: {values.mean().item():.3f}")
    print(f"Std: {values.std().item():.3f}")
    print(f"Min: {values.min().item():.3f}")
    print(f"Max: {values.max().item():.3f}")

# Example usage
analyze_policy_behavior(
    model,
    "Complete this sentence: The future of artificial intelligence will",
    n_samples=3
)

Analyzing policy behavior for: Complete this sentence: The future of artificial intelligence will

Top 10 next token probabilities:
 be: 0.680
 depend: 0.199
 likely: 0.064
 continue: 0.006
 not: 0.006
 involve: 0.006
 require: 0.006
 have: 0.004
 largely: 0.004
 see: 0.003

3 different completions:

Sample 1 (avg value: -0.057):
Complete this sentence: The future of artificial intelligence will depend on the company's ability to adapt and learn.

This sentence is trying to convey a specific idea, but it could be improved for clarity and concision. Here's a revised version:

The future of artificial intelligence will depend on the company's ability to adapt and learn.

I made a few changes to achieve this:

* Removed the phrase "the future of artificial intelligence" which is a bit of a

Sample 2 (avg value: 0.058):
Complete this sentence: The future of artificial intelligence will depend heavily on its ability to learn from experience, adapt to new situations, and solve complex proble